# IMT 573 - Lab 4 - Data Integration

### Instructions

Before beginning this assignment, please ensure you have access to a working instance of Jupyter Notebooks with Python 3.

1. First, replace the “YOUR NAME HERE” text in the next cell with your own full name. Any collaborators must also be listed in this cell.

2. Be sure to include well-documented (e.g. commented) code cells, figures, and clearly written text  explanations as necessary. Any figures should be clearly labeled and appropriately referenced within the text. Be sure that each visualization adds value to your written explanation; avoid redundancy – you do no need four different visualizations of the same pattern.

3. Collaboration on problem sets and labs is fun, useful, and encouraged. However, each student must turn in an individual write-up in their own words as well as code/work that is their own. Regardless of whether you work with others, what you turn in must be your own work; this includes code and interpretation of results. The names of all collaborators must be listed on each assignment. Do not copy-and-paste from other students’ responses or code - your code should never be on any other student's screen or machine.

4. All materials and resources that you use (with the exception of lecture slides) must be appropriately referenced within your assignment.

Name:Andy Nguyen

Collaborators: 

In this module, we have focused on integrating and cleaning data. In this lab, we'll look at integrating different data sources.

The data we will use comes from the City of Seattle. It consists of police beats in the Seattle area and provides information on their geographic locations. You can learn more about police precincts and beats [here](https://www.seattle.gov/police/about-us/about-policing/precinct-and-patrol-boundaries). We'll use this same dataset in a future problem set. 

The data can be found in the `Police_Beat_and_Precinct_Centerpoints.csv` file.

In [1]:
import pandas as pd
beats_data = pd.read_csv(r"C:\Users\xz_an\OneDrive\Desktop\catalina\Police_Beat_and_Precinct_Centerpoints.csv")

In [2]:
beats_data.head()

,Name,Location 1,Latitude,Longitude
0,B1,"(47.7097756394592, -122.370990523069)",47.70978,-122.37099
1,B2,"(47.6790521901374, -122.391748391741)",47.67905,-122.39175
2,B3,"(47.6812920482227, -122.364236159741)",47.68129,-122.36424
3,C1,"(47.6342500180223, -122.315684762418)",47.63425,-122.31568
4,C2,"(47.6192385752996, -122.313557430551)",47.61924,-122.31356


### Problem 1: Inspection

Inspect the beats data. How many records are there? What are the variables? Is there any missing or seemingly anomolous data?

In [3]:
num_records = len(beats_data)
print(f"Number of records: {num_records}")

Number of records: 57


In [4]:
print("Variables:", list(beats_data.columns), "\n")

Variables: ['Name', 'Location 1', 'Latitude', 'Longitude'] 



In [5]:
print("Missing values per column:")
print(beats_data.isnull().sum(), "\n")

Missing values per column:
Name          0
Location 1    0
Latitude      0
Longitude     0
dtype: int64 



In [6]:
print("Summary statistics:")
print(beats_data.describe(), "\n")

Summary statistics:
        Latitude   Longitude
count  57.000000   57.000000
mean   47.616469 -122.329209
std     0.056895    0.032597
min    47.509350 -122.400000
25%    47.575810 -122.351870
50%    47.615760 -122.329960
75%    47.658550 -122.306590
max    47.726550 -122.259540 



The dataset contains 57 records and 4 variables — Name, Location 1, Latitude, and Longitude. There are no missing values in any of the columns, and all variables appear to be correctly populated. The numeric summary shows latitude values between 47.59 and 47.73 and longitude values between –122.39 and –122.26, which are reasonable coordinates for the Seattle area.

### Problem 2: Using an API

We're going to join census data to the beats dataset. To do so, we need to first get census tract information for the beats. 

We'll use the `censusgeocode` package to get census tract information for this task. We have seen how different websites/data sources can have APIs and leverage API keys. Python also has many packages that will leverage APIs and `censusgeocode` is one such package in that it can interact with the US Census' APIs.

To start, import the `censusgeocode` package. As always, if the package does not import, you may need to install it first.

In [ ]:
!python -m pip install censusgeocode

In [ ]:
import censusgeocode as cg

Now, use the [documentation](https://pypi.org/project/censusgeocode/) from the `censusgeocode` package to write a function with the following specifications: 

- the function should accept two arguments - one for longitude and one for latitude (in that order)
- the function should return the census tract number (often coded as `GEOID`) for the inputted latitude and longitude as a string
- the function should be named `get_census_tract`

You can find example outputs below to test your function

In [ ]:
import requests

def get_census_tract(lon, lat):
    #Return the Census Tract GEOID for a given longitude and latitude.
    try:
        url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
        params = {
            "x": lon,
            "y": lat,
            "benchmark": "Public_AR_Current",
            "vintage": "Current_Current",
            "format": "json",
            "for": "tract:*"
        }

        response = requests.get(url, params=params)
        data = response.json()
        geogs = data["result"]["geographies"]

        #Try to get the GEOID from Census Tracts first
        if "Census Tracts" in geogs:
            return geogs["Census Tracts"][0]["GEOID"]
        #Otherwise use Census Blocks (first 11 digits)
        elif "Census Blocks" in geogs:
            return geogs["Census Blocks"][0]["GEOID"][:11]
        else:
            return None

    except Exception as e:
        print("Error:", e)
        return None

In [ ]:
get_census_tract(-77.036543, 38.898691) #should return '11001980000'
get_census_tract(-73.985428, 40.748817) #should return '36061007600'
get_census_tract(-118.321495, 34.134117) #should return '06037980009'

In [ ]:
print(get_census_tract(-77.036543, 38.898691))   #Washington, DC
print(get_census_tract(-73.985428, 40.748817))   #New York, NY
print(get_census_tract(-118.321495, 34.134117))  #Los Angeles, CA

### Problem 3: Get census tracts

Now, for each of the beats in the beats dataset, find the associated census tract. Keep this code as you'll use it in a future problem set.

Census tracts are codes to designate specific locations. The block codes are comprised of state/territory codes, followed by county codes, tract codes, and block codes. You can learn more about this [here](https://transition.fcc.gov/form477/Geo/more_about_census_blocks.pdf) . Confirm that each of the tracts for the beats data is from the state of Washington (code 53) and King County (the county that the city of Seattle is in - code 033).

In [ ]:
import requests
import time

In [ ]:
def get_census_tract(lon, lat):
    """Return the Census Tract GEOID for (lon, lat), or None if not found."""
    try:
        url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
        params = {
            "x": lon, "y": lat,
            "benchmark": "Public_AR_Current",
            "vintage": "Current_Current",
            "format": "json",
            "for": "tract:*"
        }
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        geogs = r.json()["result"]["geographies"]

        if geogs.get("Census Tracts"):
            return geogs["Census Tracts"][0]["GEOID"]
        if geogs.get("Census Blocks"):
            return geogs["Census Blocks"][0]["GEOID"][:11]  # fallback
    except Exception as e:
        print(f"Failed at ({lat}, {lon}): {e}")
    return None

def _to_tract(row):
    geoid = get_census_tract(row["Longitude"], row["Latitude"])
    time.sleep(0.12) 
    return geoid

In [ ]:
#Add tract for each beat
beats_data["Census_Tract"] = beats_data.apply(_to_tract, axis=1)

#Extract FIPS codes
beats_data["StateFIPS"]  = beats_data["Census_Tract"].str[:2]
beats_data["CountyFIPS"] = beats_data["Census_Tract"].str[2:5]

#Print results
print(beats_data[["Name","Census_Tract","StateFIPS","CountyFIPS"]].head())
print("Tracts retrieved:", beats_data["Census_Tract"].notna().sum(), "/", len(beats_data))
print("All WA (53)?",  (beats_data["StateFIPS"]  == "53").all())
print("All King (033)?", (beats_data["CountyFIPS"] == "033").all())